# How to Use This Notebook

This notebook is designed to be run sequentially from top to bottom without manual intervention. The cells are grouped into numbered steps. Please execute each step in order to run the verification.

- **Step 1: Environment Setup:** Prepares the Kaggle environment, installs dependencies, and verifies TPU access.
- **Step 2: Apply Compatibility Fix:** Downgrades NumPy to prevent known version conflicts.
- **Step 3: Configure Checkpoint Path:** Finds the Llama 3.1 checkpoint dataset and sets the required environment variable.
- **Step 4: Generate Config and Run Verification:** Uses the verified `load_parameters_path` key to generate the complete YAML file and runs the final 1-step training verification. Success is indicated by a "Verification run completed successfully" message.


# Step 1 & 2: Environment Setup

This step prepares the Kaggle environment by:
1.  **Verifying JAX and TPU Access:** Ensures the notebook can see the 8 TPU devices.
2.  **Cloning MaxText:** Clones the `google/maxtext` repository, which contains the training scripts.
3.  **Installing Dependencies:** Installs all Python packages required by MaxText from `requirements.txt`.


In [ ]:
# Cell 1: Complete Environment Setup
import os, sys, platform, subprocess

# --- Part 1: Clone MaxText and Install Dependencies ---
print("Verifying JAX and TPU environment...")
try:
    import jax
    print(f"✅ JAX version: {jax.__version__}")
    print(f"✅ Detected {jax.device_count()} TPU devices.")
except Exception as e:
    print(f"❌ ERROR: JAX/TPU verification failed: {e}")
    raise

print("\\nCloning MaxText repository...")
if not os.path.exists('maxtext'):
    subprocess.run(["git", "clone", "https://github.com/google/maxtext.git"], check=True)
print("✅ MaxText repository cloned.")

# Pin to MaxText commit compatible with JAX 0.4.34
stable_commit_hash = "4651cb3c73de"
print(f"\\nChecking out MaxText commit: {stable_commit_hash}")
subprocess.run(["git", "checkout", stable_commit_hash], check=True, cwd="maxtext", capture_output=True)
print("✅ Git checkout successful.")

print("\\nInstalling dependencies...")
subprocess.run(["apt-get", "update"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "pkg-config"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "maxtext/requirements.txt"], check=True, capture_output=True)
print("✅ Dependencies installed.")

# --- Part 2: Apply Compatibility Fixes ---
print("\\nApplying NumPy compatibility fix...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2"], check=True, capture_output=True)

# Verify the installed NumPy version
result_numpy = subprocess.run([sys.executable, "-c", "import numpy as np; print(np.__version__)"], check=True, capture_output=True, text=True)
numpy_version = result_numpy.stdout.strip()
print(f"✅ NumPy version is now: {numpy_version}")

print("\\nApplying Flax compatibility fix...")
# Force install the version of Flax specified in the requirements for this commit
subprocess.run([sys.executable, "-m", "pip", "install", "flax==0.8.4"], check=True, capture_output=True)

# Verify the installed Flax version
result_flax = subprocess.run([sys.executable, "-c", "import flax; print(flax.__version__)"], check=True, capture_output=True, text=True)
flax_version = result_flax.stdout.strip()
print(f"✅ Flax version is now: {flax_version}")

print("\\n✅ Full environment setup is complete.")

# ⚠️ Important: Restart the Kernel Now

Go to the **Run** menu and select **Restart session**. This is required for the NumPy version change to take effect. Do not run the cells below until you have restarted the session.

# Step 3: Configure Checkpoint Path

This step dynamically locates the pre-converted Llama 3.1 MaxText checkpoint within the attached Kaggle Datasets and sets the `MAXTEXT_CHECKPOINT_DIR` environment variable. This makes the checkpoint path available for the final verification step.


In [1]:
import os
from pathlib import Path

dataset_path = Path("/kaggle/input/llama-3-1-8b-maxtext-checkpoint")

print(f"Inspecting dataset directory: {dataset_path}")
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

required_files = ["_CHECKPOINT_METADATA", "items"]
if all((dataset_path / f).exists() for f in required_files):
    checkpoint_dir = dataset_path
    print(f"✅ Checkpoint found in root directory: {checkpoint_dir}")
else:
    raise FileNotFoundError(f"Could not find required checkpoint files in {dataset_path}")

os.environ["MAXTEXT_CHECKPOINT_DIR"] = str(checkpoint_dir)
print(f"✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR={os.environ['MAXTEXT_CHECKPOINT_DIR']}")


Inspecting dataset directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Checkpoint found in root directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR=/kaggle/input/llama-3-1-8b-maxtext-checkpoint


In [2]:
!ls -R /kaggle/input/llama-3-1-8b-maxtext-checkpoint

/kaggle/input/llama-3-1-8b-maxtext-checkpoint:
_CHECKPOINT_METADATA  items

/kaggle/input/llama-3-1-8b-maxtext-checkpoint/items:
_METADATA  _sharding  array_metadatas  d  manifest.ocdbt  ocdbt.process_0

/kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/array_metadatas:
process_0

/kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/d:
eaf5402255b1328a6aaea5f5c29c466d

/kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/ocdbt.process_0:
d  manifest.ocdbt

/kaggle/input/llama-3-1-8b-maxtext-checkpoint/items/ocdbt.process_0/d:
12795c0027bf7653202291725ed30674  7afabf43ac91248969a89813b12b66f2
1d9283ccf8bdc7b76ea9a0d30ecac2cd  7f591ee93524235c7640b9da425366de
4ca1faba71ad168cdb2db02e6de60f89  bda1fd7d843c461e3efde3d8383c2d74
4dabe39c304dbc960321bc5ffbbd4a1a  fa03339b643368ccfeffe75580916ecb
744009c1061d4ac00b3a2e0a56899064


# Step 4: Generate Config and Run Verification

This is the final, fully automated step. It performs the following actions:

1.  **Sets `PYTHONPATH`:** Ensures the MaxText library can be correctly imported.
2.  **Generates YAML:** Creates the `verification_minimal.yml` file using the verified `load_parameters_path` key and the checkpoint path from the previous step.
3.  **Runs Verification:** Executes the MaxText training script as a module (`MaxText.train`) for a single step. 

A successful run will print "✅ Verification run completed successfully." and is the evidence that the entire environment is correctly configured.


In [4]:
import os
import sys
import runpy
from pathlib import Path
import subprocess

# Set environment variable to resolve TensorFlow protobuf conflict
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. Define paths and add MaxText to the Python path for in-process import
maxtext_repo_path = Path.cwd() / "maxtext"
maxtext_pkg_path = maxtext_repo_path / "MaxText" # Path to the actual source files

# Add the repo root for runpy to find the 'MaxText' package
if str(maxtext_repo_path) not in sys.path:
    sys.path.insert(0, str(maxtext_repo_path))
    
# Add the package path for scripts inside to find each other (e.g., train.py importing checkpointing.py)
if str(maxtext_pkg_path) not in sys.path:
    sys.path.insert(0, str(maxtext_pkg_path))

# --- Start of Patches ---

# Patch to fix the TypeError: typing.Optional
train_script_path = maxtext_pkg_path / "train.py"
print(f"Applying type hint patch to {train_script_path}...")
subprocess.run([
    "sed",
    "-i",
    "s/config: Optional\\[pyconfig.config\\]/config: Optional[pyconfig.HyperParameters]/",
    str(train_script_path)
], check=True)
print("✅ Type hint patch applied successfully.")

# --- End of Patches ---

# NEW PATCH: Modify checkpointing to handle transposed K and V weights
checkpointing_script_path = maxtext_pkg_path/"checkpointing.py"
print(f"Applying transpose patch to {checkpointing_script_path}...")
subprocess.run([
    "sed",
    "-i",
    # 2. Find the restore call and wrap it to add the transform_fn
    (
        # This new function handles both the 2D K/V weights and the 3D MLP weights
        r"s/restored = ckptr.restore(/def _transpose_fn(x):\\n  if x.ndim == 2 and x.shape[0] == 4096 and x.shape[1] == 1024:\\n    return x.transpose()\\n  elif x.ndim == 3 and x.shape == (4096, 32, 14336):\\n    return x.transpose(1, 0, 2)\\n  return x\\n"
        r"args = ocp_args.PyTreeRestore(transform_fn=_transpose_fn)\\"
        r"    restored = ckptr.restore(args=args,/"
    ),
    str(checkpointing_script_path)
], check=True)
subprocess.run([
    "sed",
    "-i",
    # 2. Find the restore call and wrap it to add the transform_fn
    (
        r"s/restored = ckptr.restore(/def _transpose_fn(x):\\n  if x.ndim == 2 and x.shape[0] == 4096 and x.shape[1] == 1024:\\n    return x.transpose()\\n  return x\\n"
        r"args = ocp_args.PyTreeRestore(transform_fn=_transpose_fn)\\n"
        r"    restored = ckptr.restore(args=args,/"
    ),
    str(checkpointing_script_path)
], check=True)
print("✅ Transpose patch applied successfully.")


print(f"✅ Added to sys.path: {maxtext_pkg_path}")

# 2. Get checkpoint path
checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run the previous step first.")

# 3. Generate the EXHAUSTIVE YAML configuration
config_text = f"""
# ======================================================================================
# Exhaustive Configuration - Brute-force solution including all possible parameters
# to prevent any further missing key errors.
# ======================================================================================

# --------------------------------------------------------------------------------------
# Core Run & Hardware Settings (Overridden for this verification run)
# --------------------------------------------------------------------------------------
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
hardware: 'tpu'
tokenizer_path: "/kaggle/input/llama-31-8b-instruct/tokenizer.model"
load_parameters_path: "{checkpoint_path}/items"
dataset_path: "gs://dummy-bucket/dummy-data" # Using dummy data
dataset_type: "synthetic"

# --------------------------------------------------------------------------------------
# Model Parameters (Set for Llama 3.1 8B)
# --------------------------------------------------------------------------------------
model_name: "llama3.1-8b"
global_parameter_scale: 1
num_experts: 1
base_emb_dim: 4096
base_num_query_heads: 32
base_num_kv_heads: 8
base_num_decoder_layers: 32
base_mlp_dim: 14336
head_dim: 128
vocab_size: 128256
max_target_length: 2048
max_prefill_predict_length: 1024
normalization_layer_epsilon: 0.00001
decoder_block: 'llama2'
enable_dropout: False
dropout_rate: 0.0
logits_via_embedding: False
mlp_activations: ['silu', 'linear']
dtype: "bfloat16"
weight_dtype: "bfloat16"
attn_logits_soft_cap: 0
final_logits_soft_cap: 0
rope_max_timescale: 500000
use_iota_embed: False
use_untrainable_positional_embedding: False
trainable_position_size: -1
rope_min_timescale: 1
logits_dot_in_fp32: False
fused_qkv: False
fused_mlp: False
activations_in_float32: False

# --------------------------------------------------------------------------------------
# Parameters from full review of layers/models.py
# --------------------------------------------------------------------------------------
record_internal_nn_metrics: False
num_layers_per_pipeline_stage: 1
param_scan_axis: 0
set_remat_policy_on_layers_per_stage: False
tensors_on_device: []
tensors_to_offload: []
normalize_embedding_logits: False
matmul_precision: "default"
cast_logits_to_fp32: False
use_dpo: False
dpo_label_smoothing: 0.0
dpo_beta: 0.0

# --------------------------------------------------------------------------------------
# Parallelism & Sharding Settings (Set for single-host TPU v5e-8)
# --------------------------------------------------------------------------------------
ici_data_parallelism: 8
ici_fsdp_parallelism: 1
ici_tensor_parallelism: 1
ici_sequence_parallelism: 1
ici_pipeline_parallelism: 1
ici_autoregressive_parallelism: 1
ici_expert_parallelism: 1
ici_fsdp_transpose_parallelism: 1
dcn_data_parallelism: 1
dcn_fsdp_parallelism: 1
dcn_tensor_parallelism: 1
dcn_sequence_parallelism: 1
dcn_pipeline_parallelism: 1
dcn_autoregressive_parallelism: 1
dcn_expert_parallelism: 1
dcn_fsdp_transpose_parallelism: 1
mesh_axes: ['data', 'fsdp', 'pipeline', 'tensor', 'sequence', 'fsdp_transpose', 'expert', 'autoregressive']
logical_axis_rules: [
  ['embed', 'tensor'],
  ['mlp', 'tensor'],
  ['attention', 'tensor'],
  ['heads', 'tensor']
]
data_sharding: ['data', 'fsdp']
compute_axis_order: "0,1,2,3"
allow_split_physical_axes: False
custom_mesh: ""

# --------------------------------------------------------------------------------------
# Training & Optimizer Settings
# --------------------------------------------------------------------------------------
per_device_batch_size: 1
eval_per_device_batch_size: 1
learning_rate: 3.e-5
cosine_learning_rate_final_fraction: 0.1
warmup_steps_fraction: 0.1
learning_rate_schedule_steps: -1
opt_type: "adamw"
optimizer_memory_host_offload: False
adam_b1: 0.9
adam_b2: 0.95
adam_eps: 1.e-8
adam_eps_root: 0.0
adam_weight_decay: 0.1
gradient_clipping_threshold: 1.0
gradient_accumulation_steps: 1
remat_policy: 'full'
scan_layers: True
attention: 'dot_product'
attention_type: 'global'
enable_data_shuffling: True
data_shuffle_seed: 0
init_weights_seed: 0
model_call_mode: ''

# --------------------------------------------------------------------------------------
# Checkpointing Settings (with defaults for all keys)
# --------------------------------------------------------------------------------------
enable_checkpointing: True
async_checkpointing: False
enable_single_replica_ckpt_restoring: False
enable_emergency_checkpoint: False
checkpoint_period: 10
save_config_to_gcs: False
enable_checkpoint_cloud_logger: False
checkpoint_storage_use_ocdbt: False
checkpoint_storage_use_zarr3: False
load_full_state_path: ""

# --------------------------------------------------------------------------------------
# Quantization & Attention Variants
# --------------------------------------------------------------------------------------
quantization: ""
quantization_local_shard_count: -1
quantize_kvcache: False
kv_quant_axis: ""
use_ragged_attention: False
ragged_block_size: 256
sa_block_q: 512
sa_block_kv: 512
sa_block_kv_compute: 512
sa_block_q_dkv: 512
sa_block_kv_dkv: 512
sa_block_kv_dkv_compute: 512
sa_block_q_dq: 512
sa_block_kv_dq: 512
sa_use_fused_bwd_kernel: False
sa_q_layout: "HEAD_DIM_MINOR"
sa_k_layout: "HEAD_DIM_MINOR"
sa_v_layout: "HEAD_DIM_MINOR"

# --------------------------------------------------------------------------------------
# Logging, Profiling & Debugging
# --------------------------------------------------------------------------------------
log_period: 100
log_config: True
jax_cache_dir: "/kaggle/working/jax_cache"
jax_distributed_initialization_timeout: 300
jax_debug_log_modules: ""
jax_disable_jit: False
jax_enable_x64: False
jax_debug_nans: False
jax_profile_server: ""
profiler: ""
upload_all_profiler_results: False
skip_first_n_steps_for_profiler: 1
profiler_steps: 5
profile_cleanly: True
collect_stack_trace: False
stack_trace_to_cloud: False
stack_trace_interval_seconds: 600
max_checkify: False

# --------------------------------------------------------------------------------------
# Evaluation & Decoding
# --------------------------------------------------------------------------------------
eval_interval: -1
eval_steps: -1
target_eval_loss: 0.
prompt: "I love to"
load_from_prefill_dir: False
prefill_cache_dir: ""
autoregressive_decode_assert: ""
decode_sampling_strategy: "greedy"
decode_sampling_nucleus_p: -1
decode_sampling_top_k: 0
decode_sampling_temperature: 1.
# --------------------------------------------------------------------------------------
# Goodput & Metrics (DISABLED TO PREVENT GOOGLE CLOUD ERRORS)
# --------------------------------------------------------------------------------------
enable_goodput_recording: False
monitor_goodput: False
goodput_upload_interval_seconds: 60
enable_pathways_goodput: False
use_vertex_tensorboard: False
vertex_tensorboard_project: ""
vertex_tensorboard_region: ""
prometheus_port: 0

# --------------------------------------------------------------------------------------
# AOT Compilation
# --------------------------------------------------------------------------------------
compiled_trainstep_file: ""
compile_topology: ''
compile_topology_num_slices: -1

# --------------------------------------------------------------------------------------
# Inference & Caching
# --------------------------------------------------------------------------------------
inference_microbenchmark_prefill_lengths: "64,128,256,512,1024"
inference_microbenchmark_stages: "prefill,generate"
inference_microbenchmark_loop_iters: 10
inference_microbenchmark_log_file_path: ""
inference_metadata_file: ""
enable_model_warmup: False
stack_prefill_result_cache: False
prefill_cache_axis_order: "1,2,0,3"
ar_cache_axis_order: "1,2,0,3"
reshape_q: True

# --------------------------------------------------------------------------------------
# Misc / Uncategorized
# --------------------------------------------------------------------------------------
enable_jax_profiler: False
jax_profiler_port: 9999
enable_single_controller: False
sharding_tolerance: 1.0
expansion_factor_real_data: 1.0

"""

config_path = Path("/kaggle/working/verification_minimal.yml")
config_path.write_text(config_text)
print("✅ Wrote EXHAUSTIVE config to: /kaggle/working/verification_minimal.yml")


# 4. Run the verification script IN-PROCESS using runpy
print("\\n🚀 Running 1-step verification (in-process)...")

# Temporarily replace sys.argv for the script
original_argv = sys.argv
try:
    sys.argv = ['MaxText/train.py', str(config_path)]
    runpy.run_module('MaxText.train', run_name='__main__')
    print("\\n✅ Verification run completed successfully.")
except SystemExit as e:
    if e.code == 0:
        print("\\n✅ Verification run completed successfully (SystemExit code 0).")
    else:
        print(f"\\n❌ Verification run failed with SystemExit code: {e.code}")
except Exception as e:
    import traceback
    print(f"\\n❌ An unexpected error occurred: {e}")
    traceback.print_exc()
finally:
    # Always restore the original sys.argv
    sys.argv = original_argv

Applying type hint patch to /kaggle/working/maxtext/MaxText/train.py...
✅ Type hint patch applied successfully.
Applying transpose patch to /kaggle/working/maxtext/MaxText/checkpointing.py...
✅ Transpose patch applied successfully.
✅ Added to sys.path: /kaggle/working/maxtext/MaxText
✅ Wrote EXHAUSTIVE config to: /kaggle/working/verification_minimal.yml
\n🚀 Running 1-step verification (in-process)...
\n❌ An unexpected error occurred: unexpected character after line continuation character (checkpointing.py, line 255)


Traceback (most recent call last):
  File "/tmp/ipykernel_1761/2530290346.py", line 329, in <module>
    runpy.run_module('MaxText.train', run_name='__main__')
  File "/usr/local/lib/python3.10/runpy.py", line 227, in run_module
    return _run_code(code, {}, init_globals, run_name, mod_spec)
  File "/usr/local/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/kaggle/working/maxtext/MaxText/train.py", line 39, in <module>
    import checkpointing
  File "/kaggle/working/maxtext/MaxText/checkpointing.py", line 255
    abstract_unboxed_pre_state)
                             ^
SyntaxError: unexpected character after line continuation character
